# Organized model walkthrough
This notebook uses the reusable package rather than redefining equations. It explores the baseline trajectory, the M1--D phase portrait, and how the notebook's ROS control shifts the qualitative outcome.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from glial_crosstalk import DEFAULT_PARAMETERS, STATE_NAMES, simulate
from glial_crosstalk.plotting import phase_portrait, trajectory_figure


## Baseline dynamics
All values below are the original notebook defaults. The variables are dimensionless/arbitrary units, so plots are intended for qualitative comparison rather than clinical prediction.


In [ ]:
baseline = simulate(params=DEFAULT_PARAMETERS)
assert baseline.success, baseline.message
trajectory_figure(baseline)
plt.show()


In [ ]:
phase_portrait(baseline, x='M1', y='D')
plt.show()


## Perturbing microglial activation pressure
The original model exposes ROS as a control parameter: it increases M1 production through `p_ros` and suppresses the direct M2 production term through `i_ros`. The following comparison keeps every other parameter and the initial state fixed.


In [ ]:
ros_values = (0.0, 1.0, 100.0)
solutions = {ros: simulate(params=DEFAULT_PARAMETERS.updated(ROS=ros)) for ros in ros_values}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ros, sol in solutions.items():
    axes[0].plot(sol.t, sol.y[STATE_NAMES.index('M1')], label=f'ROS={ros:g}')
    axes[1].plot(sol.y[STATE_NAMES.index('M1')], sol.y[STATE_NAMES.index('D')], label=f'ROS={ros:g}')
axes[0].set(xlabel='Time (a.u.)', ylabel='M1 (a.u.)', title='M1 activation')
axes[1].set(xlabel='M1 (a.u.)', ylabel='D (a.u.)', title='M1–damage phase portrait')
for ax in axes: ax.legend(frameon=False)
plt.show()


The steady-state proxy below averages the final 5% of each trajectory. It is useful for checking whether a parameter change shifts the apparent attractor, but it is not a proof of mathematical stability.


In [ ]:
for ros, sol in solutions.items():
    tail = sol.y[:, int(0.95 * sol.y.shape[1]):].mean(axis=1)
    print(f'ROS={ros:g}: M1={tail[0]:.3g}, M2={tail[1]:.3g}, D={tail[8]:.3g}')


For a clean-environment run, install the project with `pip install -e .`; the original source notebook remains at `model_walkthrough_original.ipynb` for provenance.
